# 01 -- Data Preparation
Construct hypertensive cohort from NHANES 1999-2018 with linked mortality.
Define treatment (antihypertensive medication) and outcome (CV mortality).

**Inputs:** NHANES XPT files + mortality DAT files (from nhanes-cancer-survival project)  
**Outputs:** `data/processed/hypertension_cohort.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../nhanes-cancer-survival/data')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

CYCLES = [
    '1999-2000', '2001-2002', '2003-2004', '2005-2006', '2007-2008',
    '2009-2010', '2011-2012', '2013-2014', '2015-2016', '2017-2018'
]

---
## 1. Mortality parser

In [2]:
def load_mortality(cycle):
    fpath = DATA_DIR / f'{cycle}_mortality.dat'
    rows = []
    with open(fpath) as f:
        for line in f:
            raw = line.rstrip()
            if len(raw) < 16:
                continue
            seqn = int(raw[0:14].strip())
            eligstat = float(raw[14]) if raw[14] != '.' else np.nan
            mortstat = float(raw[15]) if raw[15] != '.' else np.nan
            ucod = raw[16:19].strip() if len(raw) >= 19 and raw[16:19].strip().isdigit() else np.nan
            
            permth_int = np.nan
            parts = raw.split()
            last = parts[-1] if parts else ''
            second_last = parts[-2] if len(parts) >= 2 else ''
            
            if last.isdigit() and len(last) > 3:
                mid = len(last) // 2
                permth_int = float(last[:mid])
            elif last.isdigit() and second_last.isdigit():
                permth_int = float(second_last)
            
            rows.append({
                'SEQN': seqn, 'ELIGSTAT': eligstat, 'MORTSTAT': mortstat,
                'UCOD_LEADING': ucod, 'PERMTH_INT': permth_int
            })
    return pd.DataFrame(rows)

t = load_mortality('2007-2008')
te = t[t['ELIGSTAT'] == 1]
print(f'Test: N={len(te):,}, Deaths={int((te["MORTSTAT"]==1).sum())}')

Test: N=6,219, Deaths=1126


---
## 2. Load and merge all cycles

In [3]:
def safe_cols(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()

all_cycles = []

for cycle in CYCLES:
    demo = pd.read_sas(DATA_DIR / f'{cycle}_demo.XPT', format='xport')
    bpq  = pd.read_sas(DATA_DIR / f'{cycle}_bpq.XPT', format='xport')
    bmx  = pd.read_sas(DATA_DIR / f'{cycle}_bmx.XPT', format='xport')
    diq  = pd.read_sas(DATA_DIR / f'{cycle}_diq.XPT', format='xport')
    smq  = pd.read_sas(DATA_DIR / f'{cycle}_smq.XPT', format='xport')
    ghb  = pd.read_sas(DATA_DIR / f'{cycle}_ghb.XPT', format='xport')
    mort = load_mortality(cycle)
    
    d = safe_cols(demo, ['SEQN','RIDAGEYR','RIAGENDR','RIDRETH1','DMDEDUC2','INDFMPIR'])
    b = safe_cols(bpq, ['SEQN','BPQ020','BPQ050A'])
    x = safe_cols(bmx, ['SEQN','BMXBMI'])
    q = safe_cols(diq, ['SEQN','DIQ010'])
    s = safe_cols(smq, ['SEQN','SMQ020'])
    g = safe_cols(ghb, ['SEQN','LBXGH'])
    m = safe_cols(mort, ['SEQN','ELIGSTAT','MORTSTAT','UCOD_LEADING','PERMTH_INT'])
    
    merged = d
    for right in [b, x, q, s, g, m]:
        merged = merged.merge(right, on='SEQN', how='left')
    
    merged['cycle'] = cycle
    all_cycles.append(merged)

df = pd.concat(all_cycles, ignore_index=True)
print(f'All cycles merged: {df.shape}')

All cycles merged: (101316, 17)


---
## 3. Define cohort: adults with hypertension

In [4]:
n_start = len(df)

# Adults 20+
df = df[df['RIDAGEYR'] >= 20].copy()
print(f'Adults 20+: {len(df):,} (dropped {n_start - len(df):,})')

# Eligible for mortality follow-up
n = len(df)
df = df[df['ELIGSTAT'] == 1].copy()
print(f'Mortality eligible: {len(df):,} (dropped {n - len(df):,})')

# Valid follow-up time
n = len(df)
df = df[df['PERMTH_INT'].notna() & (df['PERMTH_INT'] > 0)].copy()
print(f'Valid follow-up: {len(df):,} (dropped {n - len(df):,})')

# Hypertension: told by doctor (BPQ020 == 1)
n = len(df)
df = df[df['BPQ020'] == 1].copy()
print(f'Told high BP (hypertensive cohort): {len(df):,} (dropped {n - len(df):,})')

Adults 20+: 55,081 (dropped 46,235)
Mortality eligible: 54,945 (dropped 136)
Valid follow-up: 52,287 (dropped 2,658)
Told high BP (hypertensive cohort): 18,129 (dropped 34,158)


---
## 4. Define treatment, outcome, and covariates

In [5]:
# Treatment: taking prescribed antihypertensive (BPQ050A == 1)
df['treated'] = (df['BPQ050A'] == 1).astype(int)

# Outcome: CV mortality (UCOD_LEADING 001=heart disease, 005=cerebrovascular)
df['dead'] = (df['MORTSTAT'] == 1).astype(int)
df['cv_death'] = ((df['MORTSTAT'] == 1) & (df['UCOD_LEADING'].isin(['001', '005', '1', '5']))).astype(int)

# Follow-up in years
df['follow_up_yrs'] = df['PERMTH_INT'] / 12

# Demographics
df['age'] = df['RIDAGEYR']
df['female'] = (df['RIAGENDR'] == 2).astype(int)

race_map = {1: 'Mexican American', 2: 'Other Hispanic', 3: 'White', 4: 'Black', 5: 'Other'}
df['race'] = df['RIDRETH1'].map(race_map).fillna('Other')

# Education: recode to ordered (1-5)
df['education'] = df['DMDEDUC2'].where(df['DMDEDUC2'].between(1, 5), np.nan)

# Income: poverty-income ratio
df['pir'] = df['INDFMPIR'].where(df['INDFMPIR'] >= 0, np.nan)

# BMI
df['bmi'] = df['BMXBMI'].where(df['BMXBMI'] > 0, np.nan)

# Diabetes (told by doctor)
df['diabetes'] = (df['DIQ010'] == 1).astype(int)

# Smoking (ever smoked 100 cigarettes)
df['smoker'] = (df['SMQ020'] == 1).astype(int)

# HbA1c
df['hba1c'] = df['LBXGH'].where(df['LBXGH'] > 0, np.nan)

print(f'Treatment: {df.treated.sum():,} treated ({df.treated.mean()*100:.1f}%)')
print(f'CV deaths: {df.cv_death.sum():,} ({df.cv_death.mean()*100:.1f}%)')
print(f'All-cause deaths: {df.dead.sum():,} ({df.dead.mean()*100:.1f}%)')
print(f'Median follow-up: {df.follow_up_yrs.median():.1f} years')

Treatment: 13,649 treated (75.3%)
CV deaths: 1,641 (9.1%)
All-cause deaths: 4,822 (26.6%)
Median follow-up: 7.8 years


---
## 5. Handle missing data and finalize

In [6]:
analytic_cols = [
    'SEQN', 'cycle', 'treated', 'cv_death', 'dead', 'follow_up_yrs',
    'age', 'female', 'race', 'education', 'pir', 'bmi',
    'diabetes', 'smoker', 'hba1c'
]

cohort = df[analytic_cols].copy()
print(f'Before dropping missing: {len(cohort):,}')

miss = cohort.isnull().sum()
print('\nMissing values:')
print(miss[miss > 0])

# Impute: median for continuous, mode for categorical
for col in ['bmi', 'pir', 'hba1c', 'education']:
    cohort[col] = cohort[col].fillna(cohort[col].median())

print(f'\nAfter imputation: {len(cohort):,}, missing={cohort.isnull().sum().sum()}')

Before dropping missing: 18,129

Missing values:
education      34
pir          1670
bmi           508
hba1c         909
dtype: int64

After imputation: 18,129, missing=0


In [7]:
# Summary
print(f'Final cohort: {len(cohort):,}')
print(f'Treated: {cohort.treated.sum():,} ({cohort.treated.mean()*100:.1f}%)')
print(f'Untreated: {(cohort.treated==0).sum():,} ({(cohort.treated==0).mean()*100:.1f}%)')
print(f'CV deaths: {cohort.cv_death.sum():,}')
print(f'All-cause deaths: {cohort.dead.sum():,}')
cohort.dtypes

Final cohort: 18,129
Treated: 13,649 (75.3%)
Untreated: 4,480 (24.7%)
CV deaths: 1,641
All-cause deaths: 4,822


SEQN             float64
cycle             object
treated            int64
cv_death           int64
dead               int64
follow_up_yrs    float64
age              float64
female             int64
race              object
education        float64
pir              float64
bmi              float64
diabetes           int64
smoker             int64
hba1c            float64
dtype: object

In [8]:
cohort.to_csv(PROCESSED / 'hypertension_cohort.csv', index=False)
print(f'Saved: {(PROCESSED / "hypertension_cohort.csv").stat().st_size / 1e6:.1f} MB')

Saved: 1.4 MB
